In [10]:
import os, re, pickle
import numpy as np
import pandas as pd
from pathlib import Path
import json
from sklearn.metrics import mean_absolute_error

In [9]:
def _build_nextstep_table(df, axis_col="x", age_col="age", sid_col="sid"):
    """
    From a long df (one landmark already filtered), build next-step (t -> t+1) rows.
    Returns list of dicts: {sid, age_next, val_t, dt, y_true}
    """
    rows = []
    for sid, g in df.groupby(sid_col):
        g = g[[age_col, axis_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        ages = g[age_col].to_numpy(float)
        vals = g[axis_col].to_numpy(float)
        dts  = np.diff(ages)
        for i in range(len(vals) - 1):
            rows.append({
                "sid": sid,
                "age_next": ages[i+1],
                "val_t": vals[i],
                "dt": dts[i],
                "y_true": vals[i+1],
            })
    return pd.DataFrame(rows)

def _predict_nextstep(model, table, use_dt=True):
    """
    Predict y_{t+1} from val_t (+ dt if use_dt).
    Tries to match model's expected #features (1 or 2).
    """
    if table.empty:
        table["y_pred"] = []
        return table

    # Try to detect feature count; fall back to requested use_dt
    nfeat = getattr(model, "n_features_in_", 2 if use_dt else 1)
    if nfeat == 1:
        X = table[["val_t"]].to_numpy(float)
    else:
        # default to 2 features (val_t, dt) if available
        if use_dt and "dt" in table.columns:
            X = table[["val_t", "dt"]].to_numpy(float)
        else:
            # degrade gracefully if dt isn't used/available
            X = table[["val_t"]].to_numpy(float)

    y_pred = np.asarray(model.predict(X)).reshape(-1)
    out = table.copy()
    out["y_pred"] = y_pred
    return out

# def eval_nextxy_test(models_xy, test_df, landmark, use_dt=True):
#     """
#     Evaluate next-step MAE for x and y on the test split for a given landmark.
#     Returns metrics dict and per-axis prediction frames.
#     """
#     m_x, m_y = models_xy
#     # Filter this landmark
#     dL = test_df.loc[test_df["landmark"] == landmark].copy()
#     if dL.empty:
#         return {"test_MAE_x": np.nan, "test_MAE_y": np.nan, "test_MAE_xy": np.nan}, None, None

#     # Build next-step tables
#     t_x = _build_nextstep_table(dL, axis_col="x")
#     t_y = _build_nextstep_table(dL, axis_col="y")

#     # Predict
#     px = _predict_nextstep(m_x, t_x, use_dt=use_dt) if m_x is not None and not t_x.empty else pd.DataFrame()
#     py = _predict_nextstep(m_y, t_y, use_dt=use_dt) if m_y is not None and not t_y.empty else pd.DataFrame()

#     mae_x = mean_absolute_error(px["y_true"], px["y_pred"]) if not px.empty else np.nan
#     mae_y = mean_absolute_error(py["y_true"], py["y_pred"]) if not py.empty else np.nan
#     mae_xy = np.nanmean([mae_x, mae_y])

#     return {"test_MAE_x": float(mae_x), "test_MAE_y": float(mae_y), "test_MAE_xy": float(mae_xy)}, px, py
def _merge_xy(px: pd.DataFrame, py: pd.DataFrame, sid_col="sid"):
    """
    Robustly align x/y predictions for the *same* test pairs.
    Tries sensible keys; falls back to row index if needed.
    """
    if px.empty or py.empty:
        return pd.DataFrame()

    # Prefer to merge on these keys if present in both
    preferred = [sid_col, "age_t", "age_next", "dt"]
    keys = [k for k in preferred if k in px.columns and k in py.columns]
    if not keys:
        # fallback: align by row index
        px = px.reset_index().rename(columns={"index": "pair_id"})
        py = py.reset_index().rename(columns={"index": "pair_id"})
        keys = ["pair_id"]

    m = px.merge(py, on=keys, suffixes=("_x", "_y"), how="inner")
    return m

def _mae_2d_from_merged(m: pd.DataFrame, sid_col="sid"):
    """
    Expects columns: y_true_x, y_pred_x, y_true_y, y_pred_y (from _merge_xy).
    Returns overall 2D MAE, per-SID table, and merged df with per-row 2D errors.
    """
    if m.empty:
        return np.nan, pd.DataFrame(), m
    dx = m["y_pred_x"] - m["y_true_x"]
    dy = m["y_pred_y"] - m["y_true_y"]
    m["err2d"] = np.hypot(dx, dy)
    mae2d = float(m["err2d"].mean())

    per_sid = (
        m.groupby(sid_col)
         .agg(mae_2d=("err2d", "mean"),
              n_pairs=("err2d", "size"),
              mae_x=("y_pred_x", lambda _: np.nan),  # optional placeholders
              mae_y=("y_pred_y", lambda _: np.nan))
         .reset_index()
    )
    return mae2d, per_sid, m

# ---- Your function with minimal edits ----
def evaluate_xy(m_x, t_x, m_y, t_y, use_dt=True, sid_col="sid", round4=True):
    # Predict
    px = _predict_nextstep(m_x, t_x, use_dt=use_dt) if (m_x is not None and not t_x.empty) else pd.DataFrame()
    py = _predict_nextstep(m_y, t_y, use_dt=use_dt) if (m_y is not None and not t_y.empty) else pd.DataFrame()

    # 1D MAEs
    mae_x = mean_absolute_error(px["y_true"], px["y_pred"]) if not px.empty else np.nan
    mae_y = mean_absolute_error(py["y_true"], py["y_pred"]) if not py.empty else np.nan

    # Proper 2D MAE (Euclidean), aligned per test pair
    merged = _merge_xy(px, py, sid_col=sid_col)
    mae_2d, per_sid_2d, merged = _mae_2d_from_merged(merged, sid_col=sid_col)

    # Optional: also report a lower-bound when merge fails
    lb_2d = float(np.hypot(mae_x, mae_y)) if np.isfinite(mae_x) and np.isfinite(mae_y) else np.nan

    metrics = {
        "test_MAE_x": mae_x,
        "test_MAE_y": mae_y,
        "test_MAE_2D": mae_2d,            # true mean Euclidean error (if merge ok)
        "test_MAE_2D_LB": lb_2d,          # lower bound from axis MAEs
        "n_pairs_2D": int(merged.shape[0]) if not merged.empty else 0
    }

    if round4:
        metrics = {k: (round(v, 4) if isinstance(v, (int, float)) and np.isfinite(v) else v)
                   for k, v in metrics.items()}
        if not per_sid_2d.empty:
            per_sid_2d["mae_2d"] = per_sid_2d["mae_2d"].round(4)

    return metrics, px, py, per_sid_2d, merged


In [8]:

# --- your existing helper (kept here for completeness) ---
def predict_next_x(model, x_t, dt=None):
    if dt is None:
        X = np.array([[x_t]], float)
    else:
        X = np.array([[x_t, dt]], float)
    return float(model.predict(X)[0])

def load_model(name):
    """
    Load a model saved by save_model(...).
    Returns (model, meta_dict_or_None).
    """
    p = Path(name)
    with open(p, "rb") as f:
        bundle = pickle.load(f)
    return bundle["model"], bundle.get("meta")

# --- build skip-1 triplets: (t, t+1, t+2) ---
def build_skip1_triplets(df_long, sid_col="sid", landmark_col="landmark", age_col="age", x_col="x"):
    d = df_long[[sid_col, landmark_col, age_col, x_col]].dropna().copy()
    d[age_col] = pd.to_numeric(d[age_col], errors="coerce")
    d[x_col]   = pd.to_numeric(d[x_col], errors="coerce")
    d = d.dropna(subset=[age_col, x_col])

    rows = []
    for (sid, lmk), g in d.groupby([sid_col, landmark_col], sort=False):
        g = g.sort_values(age_col)
        ages = g[age_col].to_numpy()
        xs   = g[x_col].to_numpy()
        if len(xs) < 3: 
            continue
        dt = np.diff(ages)
        for i in range(len(xs)-2):
            dt1, dt2 = dt[i], dt[i+1]
            if not np.isfinite([xs[i], xs[i+1], xs[i+2], dt1, dt2]).all(): 
                continue
            if dt1 <= 0 or dt2 <= 0: 
                continue
            rows.append((sid, lmk, ages[i], ages[i+1], ages[i+2], xs[i], xs[i+1], xs[i+2], dt1, dt2))

    cols = [sid_col, landmark_col, f"{age_col}_t", f"{age_col}_t1", f"{age_col}_t2",
            "x_t", "x_t1", "x_t2", "dt1", "dt2"]
    return pd.DataFrame(rows, columns=cols)


# ---- new: infer (landmark, sex, axis) from filename like "u 6 apex_female_y_.pkl" ----
def infer_meta_from_filename(pkl_path: str):
    """
    Parse filenames like:
        "u 6 apex_female_y_.pkl"
        "porion_male_x.pkl"
        "u 6 cusp_both_y_.pkl"
    Returns: (landmark, sex_key, axis)
    """
    base = Path(pkl_path).name  # basename only (no folders)
    # landmark can have spaces/underscores; optional trailing "_" before .pkl
    m = re.match(r"^(.+?)_(male|female|both)_(x|y)_?\.pkl$", base, flags=re.IGNORECASE)
    if not m:
        raise ValueError(
            f"Filename must look like 'landmark_(male|female|both)_(x|y)[_].pkl', got: {base}"
        )
    landmark = m.group(1).strip()
    sex_key  = m.group(2).lower()
    axis     = m.group(3).lower()
    return landmark, sex_key, axis

def subset_for_model(df, landmark, sex_key, landmark_col="landmark", sex_col="sex"):
    df = df.copy()
    # landmark must match exactly (your data likely has names like "u 6 apex")
    df = df[df[landmark_col] == landmark]
    # sex filter
    if sex_key in ("male", "female"):
        want = "M" if sex_key == "male" else "F"
        df = df[df[sex_col].astype(str).str.upper() == want]
    # if 'both', no sex filter
    return df


# ---- core: evaluate skip-1 MAE on the filtered subset ----
def eval_skip1_mae(model, df_test, axis="x", use_dt=True,
                   sid_col="sid", landmark_col="landmark", age_col="age"):
    x_col = axis  # 'x' or 'y'
    trip = build_skip1_triplets(df_test, sid_col=sid_col, landmark_col=landmark_col, age_col=age_col, x_col=x_col)
    if trip.empty:
        raise ValueError("No (t, t+1, t+2) triplets after filtering (sex+landmark).")
    preds = []
    for _, r in trip.iterrows():
        xhat_t1 = predict_next_x(model, float(r["x_t"]), float(r["dt1"]) if use_dt else None)
        xhat_t2 = predict_next_x(model, xhat_t1,            float(r["dt2"]) if use_dt else None)
        preds.append(xhat_t2)
    trip["xhat_t2_skip1"] = preds
    mae = float(np.mean(np.abs(trip["xhat_t2_skip1"] - trip["x_t2"])))
    return mae, len(trip), trip

# ---- convenience: load model, infer meta, filter, eval ----
def eval_skip1_mae_from_file(pkl_path, df_test, use_dt=True,
                             sid_col="sid", landmark_col="landmark", sex_col="sex", age_col="age"):
    
    model, meta = load_model(pkl_path)
    landmark, sex_key, axis = infer_meta_from_filename(pkl_path)
    df_sub = subset_for_model(df_test, landmark, sex_key, landmark_col=landmark_col, sex_col=sex_col)
    mae, n, details = eval_skip1_mae(model, df_sub, axis=axis, use_dt=use_dt,
                                     sid_col=sid_col, landmark_col=landmark_col, age_col=age_col)
    print(f"[{landmark} | {sex_key} | axis={axis}]  Skip-1 MAE = {mae:.4f} over {n} triplets")
    return mae, n, details




In [7]:
# --- Imports you likely already have ---


# (re-use your helpers exactly as given)
# _build_nextstep_table, _predict_nextstep, predict_next_x, load_model,
# build_skip1_triplets, infer_meta_from_filename, subset_for_model
# ... paste/keep your definitions from your message ...

# ---------- New helpers: per-SID summaries ----------

def _per_sid_mae(df_err, target_col, pred_col, sid_col="sid"):
    """Return per-sid MAE + count."""
    if df_err.empty:
        return pd.DataFrame(columns=[sid_col, "mae", "n"])
    out = (df_err
           .assign(abs_err=lambda d: np.abs(d[target_col] - d[pred_col]))
           .groupby(sid_col)
           .agg(mae=("abs_err","mean"), n=("abs_err","size"))
           .reset_index())
    return out

def per_sid_next_from_file(pkl_path, df_test, use_dt=True,
                           sid_col="sid", landmark_col="landmark", sex_col="sex", age_col="age"):
    """
    Load model -> infer (landmark, sex_key, axis) -> subset test df ->
    build next-step table -> predict -> per-SID MAE.
    """
    model, _ = load_model(pkl_path)
    landmark, sex_key, axis = infer_meta_from_filename(pkl_path)  # uses basename internally
    df_sub = subset_for_model(df_test, landmark, sex_key, landmark_col=landmark_col, sex_col=sex_col)
    if df_sub.empty:
        return pd.DataFrame(columns=["sex","axis","landmark","sid","hop","mae","n"])

    tbl = _build_nextstep_table(df_sub, axis_col=axis, age_col=age_col, sid_col=sid_col)
    if tbl.empty:
        return pd.DataFrame(columns=["sex","axis","landmark","sid","hop","mae","n"])

    pred = _predict_nextstep(model, tbl, use_dt=use_dt)
    per_sid = _per_sid_mae(pred, target_col="y_true", pred_col="y_pred", sid_col="sid")
    per_sid["sex"] = sex_key
    per_sid["axis"] = axis
    per_sid["landmark"] = landmark
    per_sid["hop"] = "next"
    return per_sid[["sex","axis","landmark","sid","hop","mae","n"]]

def per_sid_skip1_from_file(pkl_path, df_test, use_dt=True,
                            sid_col="sid", landmark_col="landmark", sex_col="sex", age_col="age"):
    """
    Load model -> infer (landmark, sex_key, axis) -> subset test df ->
    build (t,t+1,t+2) -> 2-step roll-forward -> per-SID MAE.
    NOTE: build_skip1_triplets() returns columns named x_t/x_t1/x_t2 for BOTH axes.
    """
    model, _ = load_model(pkl_path)
    landmark, sex_key, axis = infer_meta_from_filename(pkl_path)
    df_sub = subset_for_model(df_test, landmark, sex_key, landmark_col=landmark_col, sex_col=sex_col)
    if df_sub.empty:
        return pd.DataFrame(columns=["sex","axis","landmark","sid","hop","mae","n"])

    trips = build_skip1_triplets(df_sub, sid_col=sid_col, landmark_col=landmark_col,
                                 age_col=age_col, x_col=axis)
    if trips.empty:
        return pd.DataFrame(columns=["sex","axis","landmark","sid","hop","mae","n"])

    # Safety: ensure expected cols exist (they're always 'x_*' by design)
    for c in ["x_t", "x_t2", "dt1", "dt2"]:
        if c not in trips.columns:
            raise KeyError(f"Expected column '{c}' missing in skip-1 table.")

    preds = []
    for _, r in trips.iterrows():
        # step-1: t -> t+1  (use 'x_t' even for Y axis)
        v_t  = float(r["x_t"])
        dt1  = float(r["dt1"]) if use_dt else None
        xhat_t1 = predict_next_x(model, v_t, dt1)

        # step-2: t+1 -> t+2
        dt2  = float(r["dt2"]) if use_dt else None
        xhat_t2 = predict_next_x(model, xhat_t1, dt2)
        preds.append(xhat_t2)

    trips = trips.assign(y_true=trips["x_t2"], y_pred=np.array(preds, float))
    per_sid = _per_sid_mae(trips.rename(columns={sid_col:"sid"}),
                           target_col="y_true", pred_col="y_pred", sid_col="sid")
    per_sid["sex"] = sex_key
    per_sid["axis"] = axis
    per_sid["landmark"] = landmark
    per_sid["hop"] = "skip1"
    return per_sid[["sex","axis","landmark","sid","hop","mae","n"]]



In [6]:
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

In [10]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_m, test_m     = re_splits["male"]
train_f, test_f     = re_splits["female"]

In [ ]:

# ---------- Driver: iterate your models_dir on TEST ONLY and save ----------
models_dir = "./models"

# You said: only test. Use your already-prepared test splits:
#   test_m, test_f, test_all  (DataFrames)
rows = []

for fname in os.listdir(models_dir):
    if not fname.lower().endswith(".pkl"):
        continue
    try:
        # infer meta using basename
        landmark, sex_key, axis = infer_meta_from_filename(fname)
    except ValueError:
        print(f"Skip (pattern mismatch): {fname}")
        continue

    # choose test df by sex
    if sex_key == "male":
        data = test_m
    elif sex_key == "female":
        data = test_f
    elif sex_key == "both":
        data = test_all
    else:
        continue

    pkl_path = os.path.join(models_dir, fname)
    try:
        # per-SID NEXT
        df_next = per_sid_next_from_file(pkl_path, data, use_dt=True)
        # per-SID SKIP-1
        df_skip = per_sid_skip1_from_file(pkl_path, data, use_dt=True)
        # collect
        if not df_next.empty:
            rows.append(df_next)
        if not df_skip.empty:
            rows.append(df_skip)
    except Exception as e:
        print(f"Skipping {fname}: {e}")

# Combine and save ONE tidy CSV
if rows:
    per_sid_mae = pd.concat(rows, ignore_index=True)
    # Optional: round to 4 decimals
    per_sid_mae["mae"] = per_sid_mae["mae"].round(4)
    per_sid_mae.to_csv("per_sid_test_mae_next_and_skip1.csv", index=False)
    print("Saved: per_sid_test_mae_next_and_skip1.csv")
else:
    print("No results to save.")


In [14]:
per_sid_mae["mae"] = per_sid_mae["mae"].round(4)
per_sid_mae.to_csv("per_sid_test_mae_next_and_skip1.csv", index=False)
print("Saved: per_sid_test_mae_next_and_skip1.csv")

Saved: per_sid_test_mae_next_and_skip1.csv


In [16]:
data = pd.read_csv("per_sid_test_mae_next_and_skip1.csv")
data.columns

Index(['sex', 'axis', 'landmark', 'sid', 'hop', 'mae', 'n'], dtype='object')

In [20]:

needed = {'sex','axis','landmark','sid','hop','mae','n'}
missing = needed - set(map(str, data.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

def make_tables(df):
    out = {}
    for sex in ['male','female','both']:
        for axis in ['x','y']:
            key = f'{sex}_{axis}'
            t = (df[(df['sex']==sex) & (df['axis']==axis)]
                   .copy()
                   .sort_values(['landmark','sid','hop']))
            out[key] = t
    return out

tables = make_tables(data)

# unpack to variables
male_x   = tables['male_x']
male_y   = tables['male_y']
female_x = tables['female_x']
female_y = tables['female_y']
both_x   = tables['both_x']
both_y   = tables['both_y']

# save to CSV
# male_x.to_csv("male_x_per_sid_test_mae.csv", index=False)
# male_y.to_csv("male_y_per_sid_test_mae.csv", index=False)
# female_x.to_csv("female_x_per_sid_test_mae.csv", index=False)
# female_y.to_csv("female_y_per_sid_test_mae.csv", index=False)
# both_x.to_csv("both_x_per_sid_test_mae.csv", index=False)
# both_y.to_csv("both_y_per_sid_test_mae.csv", index=False)


In [21]:
male_x

,sex,axis,landmark,sid,hop,mae,n
272,male,x,ans,003,next,1.2448,8
293,male,x,ans,003,skip1,1.3472,7
273,male,x,ans,013,next,1.0415,7
294,male,x,ans,013,skip1,1.2552,6
274,male,x,ans,02377,next,1.5171,11
...,...,...,...,...,...,...,...
8725,male,x,u_6_mcp,O012,skip1,2.2933,5
8710,male,x,u_6_mcp,O153,next,2.0700,8
8726,male,x,u_6_mcp,O153,skip1,2.0856,7
8711,male,x,u_6_mcp,O179,next,1.1403,7


In [22]:
# assumes you already built:
# male_x, male_y, female_x, female_y, both_x, both_y
# with columns: ['sex','axis','landmark','sid','hop','mae','n']

import numpy as np
import pandas as pd

def hop_to_wide(df):
    """Pivot hop -> columns for both mae and n. Index: (landmark, sid)."""
    if df.empty:
        return df.copy()
    df = df.copy()
    df['hop'] = df['hop'].str.lower()
    wide = df.pivot_table(index=['landmark','sid'],
                          columns='hop',
                          values=['mae','n'],
                          aggfunc={'mae':'mean','n':'sum'})  # mean mae, sum counts
    # Ensure both cols exist even if one hop is missing
    for col in [('mae','next'), ('mae','skip1'), ('n','next'), ('n','skip1')]:
        if col not in wide.columns:
            wide[col] = np.nan
    # Flatten multiindex columns
    wide.columns = [f"{a}_{b}" for a,b in wide.columns]
    wide = wide.reset_index()
    # Optional: overall average MAE across hops
    if 'mae_next' in wide and 'mae_skip1' in wide:
        wide['mae_avg'] = wide[['mae_next','mae_skip1']].mean(axis=1)
    # Round MAEs nicely
    for c in ['mae_next','mae_skip1','mae_avg']:
        if c in wide:
            wide[c] = wide[c].round(4)
    return wide

# Build wide versions
male_x_wide   = hop_to_wide(male_x)
male_y_wide   = hop_to_wide(male_y)
female_x_wide = hop_to_wide(female_x)
female_y_wide = hop_to_wide(female_y)
both_x_wide   = hop_to_wide(both_x)
both_y_wide   = hop_to_wide(both_y)

# Save
# male_x_wide.to_csv("male_x_per_sid_test_mae_wide.csv", index=False)
# male_y_wide.to_csv("male_y_per_sid_test_mae_wide.csv", index=False)
# female_x_wide.to_csv("female_x_per_sid_test_mae_wide.csv", index=False)
# female_y_wide.to_csv("female_y_per_sid_test_mae_wide.csv", index=False)
# both_x_wide.to_csv("both_x_per_sid_test_mae_wide.csv", index=False)
# both_y_wide.to_csv("both_y_per_sid_test_mae_wide.csv", index=False)


In [23]:
male_x_wide

,landmark,sid,mae_next,mae_skip1,n_next,n_skip1,mae_avg
0,ans,003,1.2448,1.3472,8.0,7.0,1.2960
1,ans,013,1.0415,1.2552,7.0,6.0,1.1484
2,ans,02377,1.5171,1.3366,11.0,10.0,1.4268
3,ans,B0113,1.1086,1.1255,8.0,7.0,1.1170
4,ans,B0387,0.6027,0.3348,3.0,2.0,0.4688
...,...,...,...,...,...,...,...
514,u_6_mcp,BU637,1.2293,1.2357,9.0,8.0,1.2325
515,u_6_mcp,BU706,1.2105,1.4624,8.0,7.0,1.3364
516,u_6_mcp,O012,2.1883,2.2933,6.0,5.0,2.2408
517,u_6_mcp,O153,2.0700,2.0856,8.0,7.0,2.0778


In [24]:
male_x_wide.to_csv("z_male_x_per_sid_test_mae_wide.csv", index=False)
male_y_wide.to_csv("z_male_y_per_sid_test_mae_wide.csv", index=False)
female_x_wide.to_csv("z_female_x_per_sid_test_mae_wide.csv", index=False)
female_y_wide.to_csv("z_female_y_per_sid_test_mae_wide.csv", index=False)
both_x_wide.to_csv("z_both_x_per_sid_test_mae_wide.csv", index=False)
both_y_wide.to_csv("z_both_y_per_sid_test_mae_wide.csv", index=False)

## Euclidian

In [11]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_m, test_m     = re_splits["male"]
train_f, test_f     = re_splits["female"]

In [3]:
import os, re, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

def _predict_nextstep(model, tbl, use_dt=True):
    if tbl.empty: 
        return tbl.assign(y_pred=[])
    X = tbl[["val_t","dt"]].to_numpy() if use_dt else tbl[["val_t"]].to_numpy()
    yhat = np.asarray(model.predict(X)).reshape(-1)
    return tbl.assign(y_pred=yhat)

def _pairs_from_triplets_for_skip1(tri, age_col="age"):
    return pd.DataFrame({
        "sid":      tri["sid"].to_numpy(),
        "age_next": tri[f"{age_col}_t2"].to_numpy(),  # t+2
        "val_t":    tri["x_t1"].to_numpy(),           # input at t+1
        "dt":       tri["dt2"].to_numpy(),            # step t+1->t+2
        "y_true":   tri["x_t2"].to_numpy(),           # target t+2
    })

def _merge_xy(px, py):
    if px.empty or py.empty:
        return pd.DataFrame()
    keys = [k for k in ["sid","age_next","dt"] if k in px.columns and k in py.columns]
    if not keys:  # fallback to index alignment
        px = px.reset_index().rename(columns={"index":"pair_id"})
        py = py.reset_index().rename(columns={"index":"pair_id"})
        keys = ["pair_id"]
    return px.merge(py, on=keys, suffixes=("_x","_y"), how="inner")

def _find_twin_model(models_dir, landmark, sex_key, other_axis):
    # accept optional trailing underscore before .pkl
    pat1 = f"{landmark}_{sex_key}_{other_axis}.pkl"
    pat2 = f"{landmark}_{sex_key}_{other_axis}_.pkl"
    cand = [p for p in os.listdir(models_dir) if p.lower() in {pat1.lower(), pat2.lower()}]
    return os.path.join(models_dir, cand[0]) if cand else None


In [4]:
def per_sid_2d_from_anyfile(pkl_path, models_dir, data, use_dt=True):
    """
    Returns two tidy DataFrames:
      df_next2d:  landmark, sex, sid, regime='next',  mae_2d, n_pairs, model_x, model_y
      df_skip12d: landmark, sex, sid, regime='skip1', mae_2d, n_pairs, model_x, model_y
    Skips gracefully if the twin (x or y) model is missing.
    """
    # parse this file
    landmark, sex_key, axis = infer_meta_from_filename(Path(pkl_path).name)
    other_axis = "y" if axis == "x" else "x"

    # locate twin
    twin_path = _find_twin_model(models_dir, landmark, sex_key, other_axis)
    if twin_path is None:
        return pd.DataFrame(), pd.DataFrame()  # no 2D possible

    # load both models
    m_a, _ = load_model(pkl_path)
    m_b, _ = load_model(twin_path)

    # filter test data for this landmark/sex once
    df_sub = subset_for_model(data, landmark, sex_key)
    if df_sub.empty:
        return pd.DataFrame(), pd.DataFrame()

    # ---- NEXT: build pairs for both axes, predict, merge, 2D error ----
    pairs_x = _build_nextstep_table(df_sub, axis_col="x")
    pairs_y = _build_nextstep_table(df_sub, axis_col="y")
    if pairs_x.empty or pairs_y.empty:
        df_next2d = pd.DataFrame()
    else:
        px = _predict_nextstep(m_a if axis=="x" else m_b, pairs_x, use_dt=use_dt)
        py = _predict_nextstep(m_b if axis=="x" else m_a, pairs_y, use_dt=use_dt)
        m  = _merge_xy(px, py)
        if m.empty:
            df_next2d = pd.DataFrame()
        else:
            err2d = np.hypot(m["y_pred_x"] - m["y_true_x"], m["y_pred_y"] - m["y_true_y"])
            m["err2d"] = err2d
            g = (m.groupby("sid")["err2d"].agg(mae_2d="mean", n_pairs="size").reset_index())
            g["landmark"] = landmark; g["sex"] = sex_key; g["regime"] = "next"
            g["model_x"] = Path(pkl_path).name if axis=="x" else Path(twin_path).name
            g["model_y"] = Path(twin_path).name if axis=="x" else Path(pkl_path).name
            df_next2d = g

    # ---- SKIP-1: triplets -> (t+1 -> t+2) pairs for both axes ----
    tri_x = build_skip1_triplets(df_sub, x_col="x")
    tri_y = build_skip1_triplets(df_sub, x_col="y")
    if tri_x.empty or tri_y.empty:
        df_skip12d = pd.DataFrame()
    else:
        px_s = _predict_nextstep(m_a if axis=="x" else m_b,
                                 _pairs_from_triplets_for_skip1(tri_x), use_dt=use_dt)
        py_s = _predict_nextstep(m_b if axis=="x" else m_a,
                                 _pairs_from_triplets_for_skip1(tri_y), use_dt=use_dt)
        m_s  = _merge_xy(px_s, py_s)
        if m_s.empty:
            df_skip12d = pd.DataFrame()
        else:
            err2d = np.hypot(m_s["y_pred_x"] - m_s["y_true_x"], m_s["y_pred_y"] - m_s["y_true_y"])
            m_s["err2d"] = err2d
            g = (m_s.groupby("sid")["err2d"].agg(mae_2d="mean", n_pairs="size").reset_index())
            g["landmark"] = landmark; g["sex"] = sex_key; g["regime"] = "skip1"
            g["model_x"] = Path(pkl_path).name if axis=="x" else Path(twin_path).name
            g["model_y"] = Path(twin_path).name if axis=="x" else Path(pkl_path).name
            df_skip12d = g

    # round
    for df in (df_next2d, df_skip12d):
        if not df.empty:
            df["mae_2d"] = df["mae_2d"].round(4)

    return df_next2d, df_skip12d


In [12]:
models_dir = "./models"
rows_axis = []    # your existing per-axis MAE rows
rows_2d   = []    # new: true 2-D MAE rows

for fname in os.listdir(models_dir):
    if not fname.lower().endswith(".pkl"):
        continue
    try:
        landmark, sex_key, axis = infer_meta_from_filename(fname)
    except ValueError:
        print(f"Skip (pattern mismatch): {fname}")
        continue

    # choose test df by sex
    data = test_m if sex_key=="male" else (test_f if sex_key=="female" else test_all)

    pkl_path = os.path.join(models_dir, fname)
    try:
        # (1) Keep your old per-axis outputs (if you still want them)
        df_next = per_sid_next_from_file(pkl_path, data, use_dt=True)
        df_skip = per_sid_skip1_from_file(pkl_path, data, use_dt=True)
        if not df_next.empty: rows_axis.append(df_next)
        if not df_skip.empty: rows_axis.append(df_skip)

        # (2) NEW: only trigger 2D once per (landmark, sex) by starting from the 'x' file
        if axis == "x":
            df_next2d, df_skip12d = per_sid_2d_from_anyfile(pkl_path, models_dir, data, use_dt=True)
            if not df_next2d.empty: rows_2d.append(df_next2d)
            if not df_skip12d.empty: rows_2d.append(df_skip12d)

    except Exception as e:
        print(f"Skipping {fname}: {e}")

# Save per-axis (as before)
if rows_axis:
    per_sid_mae = pd.concat(rows_axis, ignore_index=True)
    per_sid_mae["mae"] = per_sid_mae["mae"].round(4)
    per_sid_mae.to_csv("per_sid_test_mae_next_and_skip1.csv", index=False)
    print("Saved: per_sid_test_mae_next_and_skip1.csv")

# Save TRUE 2-D MAE
if rows_2d:
    per_sid_2d = pd.concat(rows_2d, ignore_index=True)
    per_sid_2d.to_csv("per_sid_test_mae_next_and_skip1_2D.csv", index=False)
    print("Saved: per_sid_test_mae_next_and_skip1_2D.csv")
else:
    print("No 2-D results to save (missing twins or no overlap).")


Saved: per_sid_test_mae_next_and_skip1.csv
Saved: per_sid_test_mae_next_and_skip1_2D.csv


In [13]:
data = pd.read_csv("per_sid_test_mae_next_and_skip1_2D.csv")
data.columns

Index(['sid', 'mae_2d', 'n_pairs', 'landmark', 'sex', 'regime', 'model_x',
       'model_y'],
      dtype='object')

In [15]:
import pandas as pd

# Load
data = pd.read_csv("per_sid_test_mae_next_and_skip1_2D.csv")

# Pivot regime -> columns (next_mae_2d, skip1_mae_2d, plus counts)
wide = (
    data.set_index(["landmark", "sid", "sex", "regime"])[["mae_2d", "n_pairs"]]
        .unstack("regime")
)

# Flatten multi-index cols
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# Rename to requested names
wide = wide.rename(columns={
    "mae_2d_next": "next_mae_2d",
    "mae_2d_skip1": "skip1_mae_2d",
    "n_pairs_next": "next_n_pairs",
    "n_pairs_skip1": "skip1_n_pairs",
})

# Optional: round errors
for c in ["next_mae_2d", "skip1_mae_2d"]:
    if c in wide:
        wide[c] = wide[c].round(4)

# Split by sex and save
for sex in ["male", "female", "both"]:
    out = wide[wide["sex"] == sex].copy()
    out = out.sort_values(["landmark", "sid"])
    out.to_csv(f"per_sid_2D_{sex}.csv", index=False)
    print(f"Saved per_sid_2D_{sex}.csv with {len(out)} rows")



Saved per_sid_2D_male.csv with 519 rows
Saved per_sid_2D_female.csv with 587 rows
Saved per_sid_2D_both.csv with 1106 rows


In [4]:
import os
import pandas as pd

def per_sex_landmark_avgs_aligned(
    base_file="female_x.csv",
    input_dir=".",
    out_dir="."
):
    os.makedirs(out_dir, exist_ok=True)

    # 1) Load base order from female_x.csv
    base_path = os.path.join(input_dir, base_file)
    base = pd.read_csv(base_path)
    if "landmark" not in base.columns:
        raise ValueError(f"{base_path} must contain a 'landmark' column.")
    base_order = base["landmark"].astype(str).tolist()

    cols_needed = ["landmark", "next_mae_2d", "skip1_mae_2d"]

    # 2) Process each sex separately
    for sex in ("male", "female", "both"):
        in_path = os.path.join(input_dir, f"per_sid_2D_{sex}.csv")
        if not os.path.exists(in_path):
            print(f"[skip] {in_path} not found")
            continue

        df = pd.read_csv(in_path)
        missing = [c for c in cols_needed if c not in df.columns]
        if missing:
            raise ValueError(f"{in_path} missing columns: {missing}")

        # Average by landmark
        avg = (df[cols_needed]
               .groupby("landmark", as_index=False)
               .mean(numeric_only=True))

        # Align to base order from female_x.csv
        avg = (avg.set_index("landmark")
                   .reindex(base_order)              # keep only base landmarks, same order
                   .reset_index())

        # Round MAEs
        for c in ("next_mae_2d", "skip1_mae_2d"):
            if c in avg.columns:
                avg[c] = avg[c].round(4)

        out_path = os.path.join(out_dir, f"per_landmark_avg_2D_{sex}.csv")
        avg.to_csv(out_path, index=False)
        present = avg["next_mae_2d"].notna().sum()
        print(f"Saved {out_path} with {len(avg)} landmarks (present: {present}, missing filled as NaN)")

# Example:
# per_sex_landmark_avgs_aligned(base_file="female_x.csv", input_dir=".", out_dir=".")


In [6]:
per_sex_landmark_avgs_aligned(base_file="female_x.csv", input_dir=".", out_dir=".")

Saved .\per_landmark_avg_2D_male.csv with 26 landmarks (present: 26, missing filled as NaN)
Saved .\per_landmark_avg_2D_female.csv with 26 landmarks (present: 26, missing filled as NaN)
Saved .\per_landmark_avg_2D_both.csv with 26 landmarks (present: 26, missing filled as NaN)
